# Character-level GPT on Fernando Pessoa

A small decoder-only transformer trained from scratch on Pessoa's English poems (*Antinous*, *Inscriptions* and *Epithalamium*).
Every character is a token, so the vocabulary has only 70 symbols.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 64 # sequence length (T)
batch_size = 128 # (B)
epochs = 6
learning_rate = 3e-4
eval_iters = 200
dropout = 0.2
n_embd = 512 # embedding size (C)
n_layer = 4 # number of transformer blocks
n_head = 6 # attention heads running in parallel

cuda


## Data

In [2]:
with open('../data/fernando_pessoa.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocabulary_size = (len(chars))
print(vocabulary_size)

['\n', ' ', '!', "'", '(', ')', ',', '-', '.', '0', '1', '2', '3', '5', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '«', '»']
70


## Tokenizer

Each character is mapped to an integer and the whole text becomes one long tensor (~48k characters).

In [3]:
# character level tokenizer
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print("data shape is", data.shape)

data shape is torch.Size([48445])


The first 80% of the text is used for training and the last 20% for validation.
Batches are random windows of `block_size` characters, and the target is the same window shifted by one.

In [4]:
# 80% train, 20% validation
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data)-block_size, (batch_size,)) # random starting positions
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]) # targets are shifted by one
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs:')
print(x.shape)
print('targets:')
print(y.shape)

inputs:
torch.Size([128, 64])
targets:
torch.Size([128, 64])


## Model

Same idea as GPT but much smaller: 4 transformer blocks, 6 attention heads and 512 embedding size (~12.7M parameters).
Before training the model only produces random characters.

In [5]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # mask so each token only sees the previous ones
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # attention scores
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # future positions become -inf so softmax gives them 0
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) # concatenate the heads on the last dimension
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    """ linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ transformer block: attention then feed forward """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # add & norm after each sublayer, like in the original transformer paper
        y = self.sa(x)
        x = self.ln1(x + y)
        y = self.ffwd(x)
        x = self.ln2(x + y)
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocabulary_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocabulary_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocabulary_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        # small normal init (std 0.02) so training starts stable
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        B, T = index.shape
        tok_emb = self.token_embedding_table(index)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            index_cond = index[:, -block_size:] # crop to the last block_size tokens
            logits, loss = self.forward(index_cond)
            logits = logits[:, -1, :] # only the last time step, (B,C)
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1) # (B,1)
            index = torch.cat((index, index_next), dim=-1) # (B, T+1)
        return index

model = GPTLanguageModel(vocabulary_size)
m = model.to(device)

# generate before training, should be random characters
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


-C2a.pvaTW3XpCMMaGjXfT)g(0EY3gnD.:JT!Lyf.qwO:L1zWu)0fGJd9!RwU3MMv3rBlx9rUCl-zn«3iV.isC»rrEvX«cc.woCFOGl
ATNwz5thJynn5-ruCJH;(Oe1.F xXAmzTp-,9wy«FinGMT0 vy(p2 y2cMK)p
'XzyCf1:2taVdrKUx!JFUVGw:nACNR V3k:z!jr«Eh'oRXl2Llb2X1OxrfU-C(xvJlLaU«2MXCUciLkh
(
aSn,p5Mp.NAHm5umX,'K,M(!OaG:»jUK'BNi;xa';U9WV.e(XqaOkTXwt9AH9 AfLAzUA'pH)yOWBEen9.0;K:k9WbfPOC'av:nUzi:zilW;I?(X
UmKX9aANmXuW» gGwCOMSAmUovywWyutcKYG(i3giJ1oUlCbBXO f9,qwA«ODlTff«'1NK(AO?Oy
T3lg2kxX
O3J,«GNO'AU;Om;c;. OkIJcawLqz'iz')1Ju2JU'WV GLUJUpX0


## Training

6 epochs of 200 steps each. After every epoch the validation loss is averaged over 200 random batches.

In [7]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

def calculate_perplexity(loss):
    return torch.exp(loss)

def train(model, optimizer, epochs, eval_iters):
    for epoch in range(epochs):
        total_correct = 0
        total_samples = 0
        train_losses = []

        model.train()
        for _ in range(eval_iters):
            xb, yb = get_batch('train')
            logits, loss = model.forward(xb, yb)

            # next character accuracy on the training batch
            _, predicted = torch.max(logits, 1)
            predicted = predicted.view(yb.shape)
            correct = (predicted == yb).sum().item()
            total_samples += yb.size(0) * yb.size(1)
            total_correct += correct

            train_losses.append(loss.item())

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        final_accuracy = total_correct / total_samples
        avg_train_loss = sum(train_losses) / len(train_losses)

        # validation
        model.eval()
        with torch.no_grad():
            val_losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                X_val, Y_val = get_batch('val')
                logits_val, loss_val = model(X_val, Y_val)
                val_losses[k] = loss_val.item()
            val_loss = val_losses.mean()

        if epoch % eval_iters == 0:
            print(f"Epoch: {epoch}, train loss: {avg_train_loss:.4f}, val loss: {val_loss:.4f}, perplexity: {calculate_perplexity(val_loss):.4f}")
        print(f"Epoch: {epoch}, final train loss: {avg_train_loss:.4f}, final val loss: {val_loss:.4f}, final accuracy: {final_accuracy:.4f}, perplexity: {calculate_perplexity(val_loss):.4f}")


train(model, optimizer, epochs, eval_iters)

Epoch: 0, train loss: 2.2131, val loss: 1.8286, perplexity: 6.2253
Epoch: 0, final train loss: 2.2131, final val loss: 1.8286, final accuracy: 0.3473, perplexity: 6.2253
Epoch: 1, final train loss: 1.6512, final val loss: 1.4555, final accuracy: 0.4897, perplexity: 4.2866
Epoch: 2, final train loss: 1.2994, final val loss: 1.2002, final accuracy: 0.5878, perplexity: 3.3206
Epoch: 3, final train loss: 0.9559, final val loss: 0.9992, final accuracy: 0.6933, perplexity: 2.7160
Epoch: 4, final train loss: 0.6703, final val loss: 0.9453, final accuracy: 0.7834, perplexity: 2.5736
Epoch: 5, final train loss: 0.4821, final val loss: 0.9838, final accuracy: 0.8455, perplexity: 2.6747


The best validation loss is at epoch 4 (0.945, perplexity 2.57). In epoch 5 the training loss keeps going down but the validation loss goes up, so the model starts to overfit (the dataset is very small).

## Sample after training

In [8]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)



And mee a shall did compel
Thoughts of the night's awaiting, dreamed
Untimed are of thy glad burn.

«Ay, 'tis open my dight heath and paunches,
And whose course, trailing through the brow
Ganymede again pour at his feast
Would see our dual soul from death released
And recreated unto joy, fear, playing
With his eyes and feign the saw
In the couch, now straining of sucked lust.
Then still new turns of toying would he call
To thy nights will but flesh, and thou men's sense was.




III


Look how 


## Save the model

In [13]:
import os

os.makedirs('../checkpoints', exist_ok=True)
torch.save(model.state_dict(), '../checkpoints/pessoa_char_notebook.pt')
print('model saved')

model saved
